# Comparative reporting

Step-by-step notebook for comparing scored runs with `soda_mmqc.reporting`.

Views: **per-run dashboard** (`build_dashboard`), **Structure (Layer S)**,
**Applicability (Layer 1)** and **Matching (Layer 2)** — each with
arm/model multiplots and culprit tables.

The comparison axis is the **arm**: `pinned` is the baseline every run
writes, and `<check>@vN` is an arm where that skill moved off its manifest
pin. It replaced the prompt axis, which belonged to the prompt pipeline.

| | |
|--|--|
| **Check** | `micrograph-scale-bar` |
| **Models** | `model-a`, `model-b` |
| **Arms** | `pinned`, `micrograph-scale-bar@v2`, `micrograph-scale-bar@v3` |

This reads the committed fixtures under
`tests/fixtures/reporting_snapshots/`, so it runs without a live
evaluation tree. Point `EVALUATION_DIR` elsewhere to read a real one.

Run with the project venv (`uv sync`) from the repo root.


## Setup

In [ ]:
from __future__ import annotations

import pandas as pd

from soda_mmqc.reporting import (
    build_dashboard,
    load_evaluation_dir,
    plot_comparison_layer1,
    plot_comparison_layer2_binary,
    plot_comparison_layer2_graded,
    plot_comparison_layer_s,
    plot_mean_score_with_instances,
    show_instance_context,
    show_layer1_errors,
    show_layer2_errors,
    show_table,
    summarize_runs,
)
from soda_mmqc.reporting.styles import (
    LAYER1_TITLE,
    LAYER2_BINARY_TITLE,
    LAYER2_GRADED_TITLE,
    LAYER_S_TITLE,
    MEAN_SCORE_PLOT_TITLE,
)

CHECKLIST = "fig-checklist"
CHECK = "micrograph-scale-bar"
GPT_5_MODEL = "gpt-5"
GPT_5_MINI_MODEL = "gpt-5-mini-2025-08-07"
MODELS = [GPT_5_MINI_MODEL, GPT_5_MODEL]
ARMS = ["pinned", "micrograph-scale-bar@v2", "micrograph-scale-bar@v3", "arm.4"]

LAYER_S_FIG_HEIGHT = 300
LAYER_S_FIG_WIDTH = 500
LAYER1_FIG_HEIGHT = 700
LAYER1_FIG_WIDTH = 1200
LAYER2_FIG_HEIGHT = 500
LAYER2_FIG_WIDTH = 1200
DASHBOARD_FIG_HEIGHT = 300
DASHBOARD_FIG_WIDTH = 1200
MEAN_SCORE_FIG_HEIGHT = 450
MEAN_SCORE_FIG_WIDTH = 800


### Load runs and summarize

Load all `(model, arm)` pairs we will need for arm and model contrasts. One `RunSummary` per pair. Layer S counts live in `by_list_row_counts` (keyed by predictive list, usually `outputs`).

In [ ]:
runs = load_evaluation_dir(
    CHECKLIST,
    CHECK,
    models=MODELS,
    arms=ARMS,
)
summaries = summarize_runs(runs)

pd.DataFrame(
    [
        {
            "model": run.model,
            "arm": run.arm,
            "records": len(run.records),
            "by_list_keys": ", ".join(
                summaries[run.model, run.arm].by_list_keys
            ),
        }
        for run in runs
    ]
)

## Per-run dashboard (`build_dashboard`)

Single four-panel Plotly figure for one `(MODEL, ARM)` pair: Layer S counts, then Layer 1 / Layer 2 binary / Layer 2 graded stacked bars by field.

Use this as a quick snapshot before the comparison grids below. It does **not** overlay arms or models — see whether that one-glance view is still worth keeping alongside the step-by-step sections.

In [ ]:
(
    build_dashboard(summaries[GPT_5_MODEL, ARMS[0]])
    .update_layout(
        width=DASHBOARD_FIG_WIDTH,
        height=500,
        autosize=True,
        margin=dict(l=40, r=20, t=50, b=40),
    )
)

## Structural reporting

Reporting whether the lists of objects, ie. the panels, are correct, missing or spurious on this check.

In [ ]:
fig_layer_s = plot_comparison_layer_s(
    summaries,
    compare="arm",
    model=GPT_5_MODEL,
)
if fig_layer_s is None:
    print(f"No {LAYER_S_TITLE} data for model={MODEL}")
else:
    fig_layer_s.update_layout(
        height=LAYER_S_FIG_HEIGHT,
        width=LAYER_S_FIG_WIDTH,
        autosize=True,
        margin=dict(l=40, r=20, t=50, b=40),
    )
    fig_layer_s.show()

## Applicability reporting (Layer 1)

Per leaf field: stacked applicability outcomes. One subplot per field; arms or models on the x-axis within each panel.

### Comparing arms

Subplot grid for `MODEL` (default `gpt-5`): three stacked bars per panel (one per arm, distinguished by **opacity**).

In [ ]:
fig_layer1_arm = plot_comparison_layer1(
    summaries,
    compare="arm",
    model=GPT_5_MODEL,
)
fig_layer1_arm.update_layout(
    height=LAYER1_FIG_HEIGHT,
    width=LAYER1_FIG_WIDTH,
    autosize=True,
)
fig_layer1_arm.show()

### Comparing models

Same layout for `ARM` (default `micrograph-scale-bar@v2`). Models on the x-axis; distinguished by **bar hatching**.

In [ ]:
fig_layer1_model = plot_comparison_layer1(
    summaries,
    compare="model",
    arm=ARMS[0],
)
fig_layer1_model.update_layout(
    height=LAYER1_FIG_HEIGHT,
    width=LAYER1_FIG_WIDTH,
    autosize=True,
)
fig_layer1_model.show()

### Applicability culprits

Instance table for one `(MODEL, ARM)` pair. Pre-filtered to `CULPRIT_LAYER1`; adjust filters in the table footer.

In [ ]:
summary_selected = summaries[GPT_5_MODEL, ARMS[2]]

show_layer1_errors(
    summary_selected,
    layer1="spurious_applicable",
    caption=(
        f"Layer 1 culprits — {CHECK} / {GPT_5_MODEL} / {ARMS[2]} "
        f"(initial filter: spurious_applicable)"
    ),
)

## Matching reporting (Layer 2)

Discrete matching stacks (binary and graded) first; finish with the continuous **mean score** view (bars + instance scatter).


### Matching-binary — comparing arms

Subplot grid for `MODEL`: stacked TP/TN/FP/FN; arms distinguished by **opacity**.

In [ ]:
fig_layer2_binary_arm = plot_comparison_layer2_binary(
    summaries,
    compare="arm",
    model=GPT_5_MODEL,
)
fig_layer2_binary_arm.update_layout(
    height=LAYER2_FIG_HEIGHT,
    width=LAYER2_FIG_WIDTH,
    autosize=True,
)
fig_layer2_binary_arm.show()

### Matching-binary — comparing models

Same layout for `ARM`; models distinguished by **bar hatching**.

In [ ]:
fig_layer2_binary_model = plot_comparison_layer2_binary(
    summaries,
    compare="model",
    arm=ARMS[3],
)
fig_layer2_binary_model.update_layout(
    height=LAYER2_FIG_HEIGHT,
    width=LAYER2_FIG_WIDTH,
    autosize=True,
)
fig_layer2_binary_model.show()

### Matching-graded — comparing arms

Graded string fields only (`match` / `mismatch` stacks).

In [ ]:
fig_layer2_graded_arm = plot_comparison_layer2_graded(
    summaries,
    compare="arm",
    model=GPT_5_MODEL,
)
fig_layer2_graded_arm.update_layout(
    height=LAYER2_FIG_HEIGHT,
    width=LAYER2_FIG_WIDTH,
    autosize=True,
)
fig_layer2_graded_arm.show()

### Matching-graded — comparing models

In [ ]:
fig_layer2_graded_model = plot_comparison_layer2_graded(
    summaries,
    compare="model",
    arm=ARMS[3],
)
fig_layer2_graded_model.update_layout(
    height=LAYER2_FIG_HEIGHT,
    width=LAYER2_FIG_WIDTH,
    autosize=True,
)
fig_layer2_graded_model.show()

### Layer 2 mean scores (`plot_mean_score_with_instances`)

Bar height = `mean_score` per leaf field (average over applicable instances only). Overlaid markers = individual instance `score`s with jitter so dense runs remain readable. Hover for `doc_id`, path, and Layer 2 label when present.


In [ ]:
summary = summaries[GPT_5_MODEL, ARMS[2]]
fig_mean_score = plot_mean_score_with_instances(
    summary,
    title=f"{CHECK} — {summary.model} / {summary.arm} — {MEAN_SCORE_PLOT_TITLE}",
)
fig_mean_score.update_layout(
    height=MEAN_SCORE_FIG_HEIGHT,
    width=MEAN_SCORE_FIG_WIDTH,
    autosize=True,
    margin=dict(l=40, r=20, t=50, b=40),
)
fig_mean_score.show()


### Matching culprits

Instance-level layer-1 outliers and layer-2 errors for one `(MODEL, ARM)` pair.


In [ ]:
arm = ARMS[2]
show_layer2_errors(
    summaries[GPT_5_MODEL, arm],
    caption=f"Layer 2 culprits — {CHECK} / {GPT_5_MINI_MODEL} / {arm}",
)

## Instance drill-down

Pick a row from a culprit table (`source`, `path`, `leaf_property`) — or set only `source` to see the full model output for that example call (caption, figure, gold vs pred JSON).

In [ ]:
# Selectors — copy from a culprit table row, or set only SOURCE for the full example
DRILL_SOURCE = "10.1038_s44319-025-00631-1/content/4"
DRILL_OBJECT_PATH = "outputs[0]"  # panel row within this source's outputs[]
DRILL_LEAF = "scale_bar_on_image"

show_instance_context(
    summaries[GPT_5_MODEL, ARMS[2]],
    source=DRILL_SOURCE,
    object_path=DRILL_OBJECT_PATH,
    leaf=DRILL_LEAF,
    figure_height=1200,
    figure_width=800,
)